**Installing Dependencies & Imports**

In [2]:
!pip install -q datasets sentence-transformers faiss-cpu requests tqdm


In [3]:
import os
import json
import faiss
import textwrap
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from typing import List, Dict, Tuple, Optional
import requests

2025-11-13 16:47:47.116366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763052467.315562     112 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763052467.380263     112 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

**Kaggle secrets**

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✅ Groq API key loaded from secrets")
else:
    print("⚠️ Groq API key not found! Using mock responses instead.")


✅ Groq API key loaded from secrets


**Load HF dataset and convert to QA format**

In [5]:
def load_qa_from_hf(dataset_id: str, split: str = "train") -> List[Dict[str,str]]:
    ds = load_dataset(dataset_id, split=split)
    qa_pairs = []
    for item in tqdm(ds):
        q = item.get("issue_category_sub_category") or "General Customer Support"
        a = item.get("conversation") or ""
        if q and a:
            qa_pairs.append({"question": str(q), "answer": str(a)})
    return qa_pairs

**Chunking utility**

In [6]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = max(end - overlap, end)
    return chunks

**Build corpus**

In [7]:
def build_corpus(qa_pairs: List[Dict[str,str]], chunk_size: Optional[int] = 400) -> Tuple[List[str], List[dict]]:
    texts, metadatas = [], []
    for idx, qa in enumerate(qa_pairs):
        q = qa['question']
        a = qa['answer']
        chunks = chunk_text(a, chunk_size) if chunk_size else [a]
        for c_idx, chunk in enumerate(chunks):
            texts.append(f"Q: {q}\nA: {chunk}")
            metadatas.append({
                "id": f"{idx}_{c_idx}",
                "question": q,
                "answer_chunk": chunk,
                "original_answer": a,
                "chunk_id": c_idx
            })
    return texts, metadatas

**Embedder**

In [8]:
class Embedder:
    def __init__(self, model_name="all-MiniLM-L6-v2", device="cpu"):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name, device=device)

    def embed_texts(self, texts: List[str], batch_size=64) -> np.ndarray:
        embeddings = self.model.encode(texts, show_progress_bar=True, batch_size=batch_size, convert_to_numpy=True)
        return embeddings.astype(np.float32)


**FAISS index**

In [9]:
class FaissIndex:
    def __init__(self, embedding_dim):
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add(self, vectors: np.ndarray, metadatas: List[dict], normalize=True):
        if normalize:
            faiss.normalize_L2(vectors)
        self.index.add(vectors)
        self.metadata.extend(metadatas)

    def search(self, query_vector: np.ndarray, top_k=5, normalize=True):
        if normalize:
            qv = query_vector.copy()
            faiss.normalize_L2(qv)
        else:
            qv = query_vector
        D, I = self.index.search(qv, top_k)
        results = []
        for score, idx in zip(D[0], I[0]):
            if idx < 0: continue
            results.append((self.metadata[idx], float(score)))
        return results

**Groq API call (mock fallback)**

In [10]:
def call_groq(prompt: str, model_name: str = "smol-groq-model"):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        # Mock fallback for Kaggle / testing
        return "⚠️ Mock answer: Groq API key not set or unreachable. Here's where the AI would answer based on retrieved context."
    
    url = "https://api.groq.ai/v1/responses"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {"model": model_name, "input": prompt}

    r = requests.post(url, headers=headers, json=payload)
    r.raise_for_status()
    data = r.json()
    return data["output"][0]["content"]

**Retrieval & Prompt**

In [11]:
def retrieve(query: str, embedder: Embedder, faiss_index: FaissIndex, top_k=5):
    q_emb = embedder.embed_texts([query])
    return faiss_index.search(q_emb, top_k)

def build_prompt(retrieved: List[Tuple[dict,float]], user_question: str, max_context_snippets=5):
    header = "You are an AI assistant for customer support. Use ONLY the provided context snippets to answer the question.\n\n"
    ctx_parts = []
    for i, (meta, score) in enumerate(retrieved[:max_context_snippets],1):
        snippet = meta.get("answer_chunk","")
        q = meta.get("question","")
        ctx_parts.append(f"Context {i} (source question: {q}):\n{snippet}\n")
    context_str = "\n---\n".join(ctx_parts) if ctx_parts else "No context retrieved.\n"
    return f"{header}Context:\n{context_str}\nUser Question: {user_question}\nAnswer:"


**Chat loop**

In [12]:
def chat_loop(embedder: Embedder, faiss_index: FaissIndex, top_k=3, max_context_snippets=5):
    print("Start chatting with the AI (type 'exit' to quit)")
    while True:
        user_q = input("\nYou: ").strip()
        if user_q.lower() in ["exit","quit"]: break
        retrieved = retrieve(user_q, embedder, faiss_index, top_k)
        for i,(m,s) in enumerate(retrieved,1):
            print(f"{i}. score={s:.4f} | source Q: {m.get('question')}\nSnippet: {textwrap.shorten(m.get('answer_chunk',''), width=150)}\n")
        prompt = build_prompt(retrieved, user_q, max_context_snippets)
        answer = call_groq(prompt)
        print("\nAssistant:", answer)

**Build FAISS index from HF dataset**

In [13]:
def build_and_save_index_from_hf(dataset_id: str, index_out_prefix="hf_faiss", chunk_size=400):
    qa_pairs = load_qa_from_hf(dataset_id)
    texts, metadatas = build_corpus(qa_pairs, chunk_size=chunk_size)
    embedder = Embedder()
    vectors = embedder.embed_texts(texts)
    dim = vectors.shape[1]
    fi = FaissIndex(dim)
    fi.add(vectors, metadatas)
    # optionally save index
    fi_save_path = f"{index_out_prefix}.index"
    meta_save_path = f"{index_out_prefix}_meta.json"
    faiss.write_index(fi.index, fi_save_path)
    with open(meta_save_path,"w") as f: json.dump(fi.metadata,f,indent=2)
    print(f"Saved index: {fi_save_path} & metadata: {meta_save_path}")
    return embedder, fi

**Main Execution**

In [14]:
if __name__ == "__main__":
    hf_dataset_id = "NebulaByte/E-Commerce_Customer_Support_Conversations"
    embedder, faiss_idx = build_and_save_index_from_hf(hf_dataset_id, index_out_prefix="ecommerce_faiss", chunk_size=400)
    print("\nTo start chat loop, call:")
    print("chat_loop(embedder, faiss_idx, top_k=3)\n")

README.md:   0%|          | 0.00/950 [00:00<?, ?B/s]

data/train-00000-of-00001-a5a7c6e4bb30b0(…):   0%|          | 0.00/827k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

100%|██████████| 1000/1000 [00:00<00:00, 14458.98it/s]

Loading embedding model: all-MiniLM-L6-v2


...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Saved index: ecommerce_faiss.index & metadata: ecommerce_faiss_meta.json

To start chat loop, call:
chat_loop(embedder, faiss_idx, top_k=3)



In [17]:
def call_groq(prompt: str, model_name: str = "smol-groq-model"):
    # Mock response for Kaggle
    return "⚠️ Mock answer: Groq API not reachable. This is where the AI would answer based on retrieved context."


In [18]:
chat_loop(embedder, faiss_idx, top_k=3)

Start chatting with the AI (type 'exit' to quit)



You:  I received a damaged product, what should I do?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. score=0.5733 | source Q: Returns and Refunds -> Damaged Goods/Poor Packaging
Snippet: check the status of your order. (After a few minutes) I see that your order was delivered two days ago. I apologize that the product arrived [...]

2. score=0.5544 | source Q: Replacement and Return Process -> Dealing with product issues after the return period
Snippet: it repaired. Customer: Okay, that's fine. Can you please guide me on how to get it repaired? Agent: Sure, we have a repair service available for [...]

3. score=0.5443 | source Q: Returns and Refunds -> Damaged Goods/Poor Packaging
Snippet: ould you please hold for a moment while I look into your order? Customer: Okay, I'll hold. Agent: Thank you for waiting. I see that you received [...]


Assistant: ⚠️ Mock answer: Groq API not reachable. This is where the AI would answer based on retrieved context.



You:  My payment failed but money was deducted. What should I do?


...

1. score=0.4936 | source Q: Invoice and Payment -> Missing invoice
Snippet: ayment method for my order? Agent: Sure. Let me check that for you. May I know the payment method you used for your order? Customer: I used my [...]

2. score=0.4601 | source Q: Return and Exchange -> Refund Delays/Complications
Snippet: . Agent: I understand how you feel, sir. We will do our best to resolve this issue as soon as possible. In the meantime, could you please [...]

3. score=0.4532 | source Q: Invoice and Payment -> Payment mode not available (e.g., Cash on Delivery)
Snippet: uide me through the payment process? Agent: Absolutely, Tom. First, please log in to your account on our website and navigate to the DSLR Camera [...]


Assistant: ⚠️ Mock answer: Groq API not reachable. This is where the AI would answer based on retrieved context.



You:  How can I apply a discount code?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. score=0.7341 | source Q: Pricing and Discounts -> Discount/Promotion Application
Snippet: nately, we cannot apply the discount code as it has expired. However, we can offer you a 10% discount on your purchase as a gesture of goodwill. [...]

2. score=0.6957 | source Q: Pricing and Discounts -> Different prices for the same product
Snippet: fund for the price difference as the seller's pricing is beyond our control. However, I can offer you a discount code that you can use on your [...]

3. score=0.6750 | source Q: Pricing and Discounts -> Discount/Promotion Application
Snippet: h a coupon code that you can use for your next purchase. Would that be okay with you? Customer: I guess that's better than nothing. What's the [...]


Assistant: ⚠️ Mock answer: Groq API not reachable. This is where the AI would answer based on retrieved context.



You:  quit
